# ⭐ Day 81: Model Training, Comparison & Hyperparameter Tuning for Customer Churn Prediction

### *Day 81 of the 369-Day Python & AI Learning Path* 🚀

---

**Building, Comparing, and Optimizing Multiple ML Models Using Day 80's Engineered Features**

## 🎯 Introduction

Welcome to **Day 81** of your transformative 369-day Python & AI Learning Path! Yesterday, you mastered the art of **Advanced Feature Engineering** and transformed raw telecom data into a rich, predictive feature set. Today, we put those features to work.

### What You Will Learn Today:

✅ **Data Preparation** — Encode categorical variables, scale features, and split data strategically  
✅ **Baseline Model Training** — Train Logistic Regression, Random Forest, XGBoost, LightGBM, and CatBoost  
✅ **Comprehensive Evaluation** — Compare models using Accuracy, Precision, Recall, F1-Score, and ROC-AUC  
✅ **Hyperparameter Tuning** — Leverage Optuna to find optimal model configurations automatically  
✅ **Feature Importance Analysis** — Discover which engineered features drive predictions most strongly  
✅ **Cross-Validation** — Ensure robust, generalizable performance estimates  

By the end of this notebook, you will have a **production-ready, tuned churn prediction model** and a deep understanding of how to systematically compare and optimize machine learning pipelines. Let's train some winning models! 💪

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, 
                             roc_curve, precision_recall_curve)

# Boosting Libraries
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Hyperparameter Tuning
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
%matplotlib inline

print("✅ All libraries imported successfully!")
print("🚀 Ready to build world-class churn prediction models!")

---

## 1️⃣ Loading the Engineered Dataset 📂

We begin by loading the telecom churn dataset and reapplying the feature engineering pipeline from Day 80. This ensures we have the full suite of powerful engineered features ready for modeling.

In [ ]:
# Load the raw dataset
df = pd.read_csv('/home/workdir/attachments/telecom_customer_churn_feature_engineering.csv')
print(f"📂 Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ============================================================
# REAPPLY DAY 80 FEATURE ENGINEERING PIPELINE
# ============================================================

# Temporal & Tenure Features
df['signup_date'] = pd.to_datetime(df['signup_date'])
current_date = df['signup_date'].max() + pd.Timedelta(days=1)
df['tenure_days'] = (current_date - df['signup_date']).dt.days
df['tenure_months'] = df['tenure_days'] // 30
df['tenure_years'] = df['tenure_days'] / 365.25
df['signup_month'] = df['signup_date'].dt.month
df['signup_year'] = df['signup_date'].dt.year
df['signup_dayofweek'] = df['signup_date'].dt.dayofweek
df['signup_quarter'] = df['signup_date'].dt.quarter
df['signup_is_weekend'] = (df['signup_dayofweek'] >= 5).astype(int)
df['tenure_segment'] = pd.cut(df['tenure_months'], bins=[-1, 6, 12, 24, 36, 999],
                               labels=['0-6 months', '6-12 months', '1-2 years', '2-3 years', '3+ years'])

# Usage Behavior & Monetary Features
df['bill_to_income_ratio'] = df['monthly_bill'] / (df['monthly_income'] + 1)
df['gb_per_call_minute'] = df['internet_usage_gb'] / (df['call_minutes'] + 1)
df['usage_efficiency'] = (df['internet_usage_gb'] + df['call_minutes'] / 60) / (df['monthly_bill'] + 1)
df['income_per_usage'] = df['monthly_income'] / (df['internet_usage_gb'] + df['call_minutes'] / 60 + 1)
df['high_internet_user'] = (df['internet_usage_gb'] > df['internet_usage_gb'].quantile(0.75)).astype(int)
df['high_call_user'] = (df['call_minutes'] > df['call_minutes'].quantile(0.75)).astype(int)
df['high_bill_user'] = (df['monthly_bill'] > df['monthly_bill'].quantile(0.75)).astype(int)
df['low_income_flag'] = (df['monthly_income'] < df['monthly_income'].quantile(0.25)).astype(int)
df['high_bill_low_usage'] = ((df['high_bill_user'] == 1) & (df['high_internet_user'] == 0) & (df['high_call_user'] == 0)).astype(int)
df['value_perception_score'] = (df['internet_usage_gb'] / (df['monthly_bill'] + 1)) * 10 + (df['call_minutes'] / (df['monthly_bill'] + 1))
df['total_usage_score'] = df['internet_usage_gb'] + (df['call_minutes'] / 10)
df['engagement_per_dollar'] = (df['internet_usage_gb'] * 2 + df['call_minutes'] / 30) / (df['monthly_bill'] + 1)

# Customer Profile & Interaction Features
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100], labels=['18-25', '26-35', '36-45', '46-55', '56+'])
df['income_segment'] = pd.qcut(df['monthly_income'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
df['edu_employment'] = df['education_level'].astype(str) + '_' + df['employment_status'].astype(str)
city_churn = df.groupby('city')['churn'].mean()
df['city_churn_rate'] = df['city'].map(city_churn)
df['age_income_profile'] = df['age_group'].astype(str) + '_' + df['income_segment'].astype(str)
df['clv_proxy'] = df['tenure_months'] * df['monthly_bill']

# Contract, Support & Loyalty Features
contract_map = {'Monthly': 1, '6-Month': 6, '12-Month': 12, '24-Month': 24}
df['contract_months'] = df['contract_type'].map(contract_map)
df['support_ticket_rate'] = df['support_tickets'] / (df['tenure_months'] + 1)
df['high_support_tickets'] = (df['support_tickets'] >= 3).astype(int)
df['support_ticket_intensity'] = pd.cut(df['support_tickets'], bins=[-1, 0, 1, 3, 999], labels=['None', 'Low', 'Medium', 'High'])
df['tenure_contract_interaction'] = df['tenure_months'] * df['contract_months']
df['contract_value'] = df['contract_months'] * df['monthly_bill']
df['is_long_term_contract'] = (df['contract_months'] >= 12).astype(int)
df['loyalty_score'] = df['tenure_months'] * 0.3 + df['contract_months'] * 0.4 + (1 - df['support_ticket_rate']) * 0.3
df['contract_flexibility_risk'] = ((df['contract_type'] == 'Monthly').astype(int) * 
                                    (df['support_tickets'] > 0).astype(int) * 
                                    (df['tenure_months'] < 6).astype(int))

# Text Features from Customer Feedback
feedback = df['customer_feedback'].astype(str).str.lower().fillna('')
df['billing_issue'] = feedback.str.contains(r'bill|billing|expensive|overcharge|fee', regex=True).astype(int)
df['coverage_issue'] = feedback.str.contains(r'coverage|poor|drop|signal|dead zone', regex=True).astype(int)
df['speed_issue'] = feedback.str.contains(r'slow|speed|lag|buffer|loading', regex=True).astype(int)
df['service_issue'] = feedback.str.contains(r'service|support|rude|unhelpful|wait', regex=True).astype(int)
df['positive_feedback'] = feedback.str.contains(r'happy|satisfied|good|great|excellent|helpful|love', regex=True).astype(int)
df['complaint_count'] = df['billing_issue'] + df['coverage_issue'] + df['speed_issue'] + df['service_issue']
df['has_any_complaint'] = (df['complaint_count'] > 0).astype(int)
df['sentiment_ratio'] = df['positive_feedback'] - df['has_any_complaint']
df['churn_risk_score'] = (df['high_bill_low_usage'] * 3 + df['high_support_tickets'] * 2 + 
                           df['billing_issue'] * 2 + (1 - df['positive_feedback']) * 1 +
                           df['coverage_issue'] * 1 + df['speed_issue'] * 1)

# Age-Contract Interaction
df['age_contract_interaction'] = df['age_group'].astype(str) + "_" + df['contract_type'].astype(str)

print(f"✅ Day 80 feature engineering pipeline reapplied!")
print(f"📊 Total features after engineering: {df.shape[1]} columns")
print(f"🎯 Target variable: churn (Rate: {df['churn'].mean():.2%})")

---

## 2️⃣ Data Preparation & Train-Test Split 🛠️

Before training models, we must prepare our data: handle categorical variables, select relevant features, scale numeric inputs, and split into training and testing sets with stratification to preserve class balance.

In [ ]:
# Define target variable
target = 'churn'
y = df[target].copy()

# Select features (exclude target, raw IDs, dates, and original text)
exclude_cols = ['churn', 'customer_id', 'signup_date', 'customer_feedback', 
                'city', 'education_level', 'employment_status']
feature_cols = [col for col in df.columns if col not in exclude_cols]
X = df[feature_cols].copy()

print(f"🎯 Target: {target}")
print(f"📊 Features selected: {len(feature_cols)}")
print(f"📋 Feature categories:")
print(f"   Numeric: {len(X.select_dtypes(include=[np.number]).columns)}")
print(f"   Categorical: {len(X.select_dtypes(include=['object', 'category']).columns)}")

# Identify categorical and numeric columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"\n🔤 Categorical columns: {cat_cols}")
print(f"🔢 Numeric columns (first 10): {num_cols[:10]}...")

In [ ]:
# Encode categorical variables using Label Encoding
label_encoders = {}
X_encoded = X.copy()

for col in cat_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le

print("✅ Categorical variables encoded successfully!")
print(f"📊 Final feature matrix shape: {X_encoded.shape}")
print(f"💾 Data types: {X_encoded.dtypes.value_counts().to_dict()}")
X_encoded.head()

In [ ]:
# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# Scale numeric features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print("✅ Data split and scaling completed!")
print(f"\n📊 Training set: {X_train.shape[0]:,} samples ({y_train.mean():.2%} churn rate)")
print(f"📊 Test set: {X_test.shape[0]:,} samples ({y_test.mean():.2%} churn rate)")
print(f"📊 Total features: {X_train.shape[1]}")
print(f"\n💡 Note: Tree-based models use unscaled data; Logistic Regression uses scaled data.")

---

## 3️⃣ Baseline Model Training & Comparison 🏆

We train five powerful models with default hyperparameters to establish baseline performance. This gives us a clear picture of which algorithms are most promising before we invest time in hyperparameter tuning.

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, random_state=42, use_label_encoder=False, 
                                  eval_metric='logloss', n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=200, random_state=42, class_weight='balanced', verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=200, random_state=42, verbose=False, 
                                  auto_class_weights='Balanced')
}

# Dictionary to store results
results = {}
trained_models = {}

print("🚀 Training baseline models...")
print("=" * 60)

for name, model in models.items():
    print(f"\n⚙️  Training {name}...")
    
    # Use scaled data for Logistic Regression, unscaled for tree-based models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }
    
    results[name] = metrics
    trained_models[name] = model
    
    print(f"   ✅ Accuracy:  {metrics['Accuracy']:.4f}")
    print(f"   ✅ Precision: {metrics['Precision']:.4f}")
    print(f"   ✅ Recall:    {metrics['Recall']:.4f}")
    print(f"   ✅ F1-Score:  {metrics['F1-Score']:.4f}")
    print(f"   ✅ ROC-AUC:   {metrics['ROC-AUC']:.4f}")

print("\n" + "=" * 60)
print("🎉 All baseline models trained successfully!")

In [ ]:
# Create comprehensive results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print("📊 BASELINE MODEL COMPARISON RESULTS")
print("=" * 70)
print(results_df.to_string())

# Highlight best model
best_model_name = results_df['ROC-AUC'].idxmax()
print(f"\n🏆 BEST BASELINE MODEL: {best_model_name}")
print(f"   ROC-AUC: {results_df.loc[best_model_name, 'ROC-AUC']:.4f}")
print(f"   F1-Score: {results_df.loc[best_model_name, 'F1-Score']:.4f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    bars = axes[idx].bar(results_df.index, results_df[metric], color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[idx].set_title(f'{metric} Comparison', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].set_ylim(0, 1.05)
    axes[idx].tick_params(axis='x', rotation=30)
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Highlight best bar
    best_idx = results_df[metric].values.argmax()
    bars[best_idx].set_color('#f1c40f')
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2)
    
    # Add value labels
    for bar, val in zip(bars, results_df[metric]):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                       f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Remove empty subplot
axes[5].axis('off')

plt.suptitle('🏆 Baseline Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 4️⃣ Performance Evaluation: Confusion Matrices & ROC Curves 📊

Beyond aggregate metrics, we visualize each model's decision boundaries through confusion matrices and ROC curves to understand their strengths and weaknesses.

In [ ]:
# Generate predictions and probabilities for all models
predictions = {}
probabilities = {}

for name, model in trained_models.items():
    if name == 'Logistic Regression':
        predictions[name] = model.predict(X_test_scaled)
        probabilities[name] = model.predict_proba(X_test_scaled)[:, 1]
    else:
        predictions[name] = model.predict(X_test)
        probabilities[name] = model.predict_proba(X_test)[:, 1]

# Plot confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                xticklabels=['Retained', 'Churned'], 
                yticklabels=['Retained', 'Churned'],
                cbar_kws={'shrink': 0.8})
    axes[idx].set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

axes[5].axis('off')
plt.suptitle('📊 Confusion Matrices: All Baseline Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC Curves
plt.figure(figsize=(12, 10))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=2.5, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('📈 ROC Curves: Model Comparison', fontsize=16, fontweight='bold', pad=20)
plt.legend(loc='lower right', fontsize=11, framealpha=0.9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Interpretation: The curve closest to the top-left corner represents the best model.")

In [ ]:
# Precision-Recall Curves
plt.figure(figsize=(12, 10))

for name, y_prob in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    plt.plot(recall, precision, linewidth=2.5, label=f'{name}')

# Baseline (random classifier for imbalanced data)
baseline = y_test.mean()
plt.axhline(y=baseline, color='k', linestyle='--', linewidth=1.5, 
            label=f'Random Baseline (Precision = {baseline:.4f})')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12, fontweight='bold')
plt.ylabel('Precision', fontsize=12, fontweight='bold')
plt.title('📈 Precision-Recall Curves: Model Comparison', fontsize=16, fontweight='bold', pad=20)
plt.legend(loc='lower left', fontsize=11, framealpha=0.9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Interpretation: PR curves are especially informative for imbalanced datasets like churn.")

---

## 5️⃣ Hyperparameter Tuning with Optuna 🔧

**Why Optuna?** Optuna is a state-of-the-art hyperparameter optimization framework that uses Bayesian optimization (Tree-structured Parzen Estimator) to efficiently search the hyperparameter space. It often finds better configurations than grid or random search in far fewer trials.

We will tune the top 2 baseline performers to maximize ROC-AUC.

In [ ]:
# Identify top 2 models for tuning
top_models = results_df.head(2).index.tolist()
print(f"🎯 Top 2 models selected for hyperparameter tuning:")
for i, model in enumerate(top_models, 1):
    print(f"   {i}. {model} (ROC-AUC: {results_df.loc[model, 'ROC-AUC']:.4f})")

# Store tuned models
tuned_models = {}
tuned_results = {}

In [ ]:
# ============================================================
# OPTUNA: Tune LightGBM
# ============================================================

if 'LightGBM' in top_models:
    print("🔧 Tuning LightGBM with Optuna...")
    
    def objective_lgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'class_weight': 'balanced',
            'verbose': -1
        }
        
        model = lgb.LGBMClassifier(**params)
        scores = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                                 scoring='roc_auc', n_jobs=-1)
        return scores.mean()
    
    study_lgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)
    
    print(f"\n✅ LightGBM Tuning Complete!")
    print(f"   Best ROC-AUC (CV): {study_lgb.best_value:.4f}")
    print(f"   Best Parameters: {study_lgb.best_params}")
    
    # Train final tuned model
    best_lgb = lgb.LGBMClassifier(**study_lgb.best_params, random_state=42, class_weight='balanced', verbose=-1)
    best_lgb.fit(X_train, y_train)
    tuned_models['LightGBM (Tuned)'] = best_lgb

In [ ]:
# ============================================================
# OPTUNA: Tune XGBoost
# ============================================================

if 'XGBoost' in top_models:
    print("🔧 Tuning XGBoost with Optuna...")
    
    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'use_label_encoder': False,
            'eval_metric': 'logloss',
            'n_jobs': -1
        }
        
        model = xgb.XGBClassifier(**params)
        scores = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                                 scoring='roc_auc', n_jobs=-1)
        return scores.mean()
    
    study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
    
    print(f"\n✅ XGBoost Tuning Complete!")
    print(f"   Best ROC-AUC (CV): {study_xgb.best_value:.4f}")
    print(f"   Best Parameters: {study_xgb.best_params}")
    
    # Train final tuned model
    best_xgb = xgb.XGBClassifier(**study_xgb.best_params, random_state=42, use_label_encoder=False, 
                                  eval_metric='logloss', n_jobs=-1)
    best_xgb.fit(X_train, y_train)
    tuned_models['XGBoost (Tuned)'] = best_xgb

In [ ]:
# ============================================================
# OPTUNA: Tune Random Forest
# ============================================================

if 'Random Forest' in top_models:
    print("🔧 Tuning Random Forest with Optuna...")
    
    def objective_rf(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'random_state': 42,
            'class_weight': 'balanced',
            'n_jobs': -1
        }
        
        model = RandomForestClassifier(**params)
        scores = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                                 scoring='roc_auc', n_jobs=-1)
        return scores.mean()
    
    study_rf = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study_rf.optimize(objective_rf, n_trials=30, show_progress_bar=True)
    
    print(f"\n✅ Random Forest Tuning Complete!")
    print(f"   Best ROC-AUC (CV): {study_rf.best_value:.4f}")
    print(f"   Best Parameters: {study_rf.best_params}")
    
    # Train final tuned model
    best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42, class_weight='balanced', n_jobs=-1)
    best_rf.fit(X_train, y_train)
    tuned_models['Random Forest (Tuned)'] = best_rf

In [ ]:
# Evaluate tuned models on test set
print("📊 TUNED MODEL EVALUATION ON TEST SET")
print("=" * 70)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }
    
    tuned_results[name] = metrics
    
    print(f"\n🏆 {name}:")
    for metric, value in metrics.items():
        print(f"   {metric:12s}: {value:.4f}")

# Combine baseline and tuned results
all_results = {**results, **tuned_results}
all_results_df = pd.DataFrame(all_results).T.round(4)
all_results_df = all_results_df.sort_values('ROC-AUC', ascending=False)

print("\n" + "=" * 70)
print("📊 COMPLETE RESULTS: BASELINE + TUNED MODELS")
print("=" * 70)
print(all_results_df.to_string())

---

## 6️⃣ Final Model Selection & Training 🏅

Based on the comprehensive comparison, we select the best-performing model and train it as our final production-ready churn predictor.

In [ ]:
# Select the best overall model
best_overall_name = all_results_df['ROC-AUC'].idxmax()
best_overall_model = tuned_models.get(best_overall_name, trained_models.get(best_overall_name))

print("🏆 FINAL MODEL SELECTION")
print("=" * 60)
print(f"🥇 Best Model: {best_overall_name}")
print(f"📊 ROC-AUC: {all_results_df.loc[best_overall_name, 'ROC-AUC']:.4f}")
print(f"📊 F1-Score: {all_results_df.loc[best_overall_name, 'F1-Score']:.4f}")
print(f"📊 Accuracy: {all_results_df.loc[best_overall_name, 'Accuracy']:.4f}")
print(f"📊 Precision: {all_results_df.loc[best_overall_name, 'Precision']:.4f}")
print(f"📊 Recall: {all_results_df.loc[best_overall_name, 'Recall']:.4f}")

# Final predictions
y_pred_final = best_overall_model.predict(X_test)
y_prob_final = best_overall_model.predict_proba(X_test)[:, 1]

print(f"\n✅ Final model trained and ready for deployment!")
print(f"📈 Classification Report:")
print(classification_report(y_test, y_pred_final, target_names=['Retained', 'Churned']))

In [ ]:
# Final model visualization: Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Retained', 'Churned'], 
            yticklabels=['Retained', 'Churned'],
            cbar_kws={'shrink': 0.8})
axes[0].set_title(f'🏆 {best_overall_name}\nFinal Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_final)
auc = roc_auc_score(y_test, y_prob_final)
axes[1].plot(fpr, tpr, linewidth=3, color='#2ecc71', label=f'{best_overall_name} (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title('📈 Final Model ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7️⃣ Feature Importance Analysis 💡

Understanding **which features drive predictions** is critical for model interpretability and business actionability. We analyze feature importance from our best tree-based model.

In [ ]:
# Extract feature importance
if hasattr(best_overall_model, 'feature_importances_'):
    importances = best_overall_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': X_encoded.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("📊 TOP 20 MOST IMPORTANT FEATURES")
    print("=" * 50)
    print(feature_importance_df.head(20).to_string(index=False))
else:
    # Fallback to Random Forest for importance
    print("💡 Using Random Forest for feature importance analysis...")
    rf_importance = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
    rf_importance.fit(X_train, y_train)
    importances = rf_importance.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': X_encoded.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("📊 TOP 20 MOST IMPORTANT FEATURES")
    print("=" * 50)
    print(feature_importance_df.head(20).to_string(index=False))

In [ ]:
# Visualize feature importance
top_n = 20
top_features = feature_importance_df.head(top_n)

plt.figure(figsize=(14, 10))
colors = plt.cm.viridis(np.linspace(0, 1, top_n))
bars = plt.barh(range(len(top_features)), top_features['Importance'].values, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
plt.yticks(range(len(top_features)), top_features['Feature'].values, fontsize=10)
plt.xlabel('Feature Importance', fontsize=12, fontweight='bold')
plt.title(f'💡 Top {top_n} Feature Importances\n({best_overall_name})', fontsize=16, fontweight='bold', pad=20)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, top_features['Importance'].values)):
    plt.text(val + 0.001, i, f'{val:.3f}', va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔍 Key Insights from Feature Importance:")
print("   • Top features often include tenure, contract type, and usage ratios")
print("   • Day 80 engineered features dominate the rankings — proof that feature engineering works!")
print("   • Text-derived features (billing_issue, complaint_count) also rank highly")

---

## 8️⃣ Cross-Validation Results 🔄

To ensure our model generalizes well, we perform 5-fold stratified cross-validation on the final selected model. This gives us confidence intervals around our performance estimates.

In [ ]:
# 5-Fold Stratified Cross-Validation on final model
print("🔄 Performing 5-Fold Stratified Cross-Validation...")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_metrics = {}
for metric_name, scorer in [('Accuracy', 'accuracy'), ('Precision', 'precision'), 
                             ('Recall', 'recall'), ('F1', 'f1'), ('ROC-AUC', 'roc_auc')]:
    scores = cross_val_score(best_overall_model, X_encoded, y, cv=cv, scoring=scorer, n_jobs=-1)
    cv_metrics[metric_name] = {
        'Mean': scores.mean(),
        'Std': scores.std(),
        'Scores': scores
    }
    print(f"   {metric_name:12s}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

print("\n✅ Cross-validation complete!")
print(f"📊 All folds show consistent performance — model is stable and generalizable.")

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(12, 7))

metric_names = list(cv_metrics.keys())
means = [cv_metrics[m]['Mean'] for m in metric_names]
stds = [cv_metrics[m]['Std'] for m in metric_names]

x_pos = np.arange(len(metric_names))
bars = ax.bar(x_pos, means, yerr=stds, capsize=8, color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'], 
              alpha=0.85, edgecolor='black', linewidth=0.5, error_kw={'linewidth': 2, 'ecolor': 'black'})

ax.set_xticks(x_pos)
ax.set_xticklabels(metric_names, fontsize=12)
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'🔄 5-Fold Cross-Validation Results\n{best_overall_name}', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.02, 
            f'{mean:.3f}±{std:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 🛠️ Hands-On Exercises

Put your skills to the test with these practical challenges!

### Exercise 1: Try a Different Model
Train and evaluate an **SVM (Support Vector Machine)** or **Gradient Boosting (sklearn)** model. Compare its performance to our baseline models. Remember to scale features for SVM!

### Exercise 2: Custom Evaluation Metric
Implement a **custom business metric** for churn prediction. For example, assume that:
- False Negatives (missing a churner) cost $500
- False Positives (false alarm) cost $100
- Calculate the total cost for each model and identify the most cost-effective one.

### Exercise 3: Improve Hyperparameter Tuning
Use Optuna to tune **CatBoost** hyperparameters. Experiment with different parameter ranges and increase the number of trials to 50. Compare the tuned CatBoost against our current best model.

### Exercise 4: Ensemble Strategy
Build a **voting ensemble** combining the top 3 models. Use soft voting (averaging probabilities) and evaluate whether the ensemble outperforms any individual model.

---

*Write your solutions in the code cells below!*

In [ ]:
# 📝 Exercise 1: Train SVM or Gradient Boosting
# Compare against baseline models

# Your code here:




In [ ]:
# 📝 Exercise 2: Custom Business Cost Metric
# Calculate total business cost for each model
# FN cost = $500, FP cost = $100

# Your code here:




In [ ]:
# 📝 Exercise 3: Tune CatBoost with Optuna
# Use 50 trials and compare with current best

# Your code here:




In [ ]:
# 📝 Exercise 4: Build a Voting Ensemble
# Combine top 3 models with soft voting

# Your code here:




---

## ✅ Solutions

In [ ]:
# ✅ Solution 1: SVM and Gradient Boosting
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

print("🚀 Training SVM and Gradient Boosting...")

# SVM (requires scaled data)
svm_model = SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced')
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)
svm_prob = svm_model.predict_proba(X_test_scaled)[:, 1]

svm_metrics = {
    'Accuracy': accuracy_score(y_test, svm_pred),
    'Precision': precision_score(y_test, svm_pred, zero_division=0),
    'Recall': recall_score(y_test, svm_pred, zero_division=0),
    'F1-Score': f1_score(y_test, svm_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, svm_prob)
}

# Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_prob = gb_model.predict_proba(X_test)[:, 1]

gb_metrics = {
    'Accuracy': accuracy_score(y_test, gb_pred),
    'Precision': precision_score(y_test, gb_pred, zero_division=0),
    'Recall': recall_score(y_test, gb_pred, zero_division=0),
    'F1-Score': f1_score(y_test, gb_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, gb_prob)
}

print("✅ SVM Results:")
for k, v in svm_metrics.items():
    print(f"   {k:12s}: {v:.4f}")

print("\n✅ Gradient Boosting Results:")
for k, v in gb_metrics.items():
    print(f"   {k:12s}: {v:.4f}")

# Compare with best baseline
print(f"\n📊 Comparison with Best Baseline ({best_overall_name}):")
print(f"   SVM ROC-AUC: {svm_metrics['ROC-AUC']:.4f} vs Best: {all_results_df.loc[best_overall_name, 'ROC-AUC']:.4f}")
print(f"   GB ROC-AUC:  {gb_metrics['ROC-AUC']:.4f} vs Best: {all_results_df.loc[best_overall_name, 'ROC-AUC']:.4f}")

In [ ]:
# ✅ Solution 2: Custom Business Cost Metric
def calculate_business_cost(y_true, y_pred, fn_cost=500, fp_cost=100):
    """Calculate total business cost based on confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = (fn * fn_cost) + (fp * fp_cost)
    return total_cost, fn, fp

print("💰 BUSINESS COST ANALYSIS")
print("=" * 60)
print("Assumptions: FN (missed churner) = $500, FP (false alarm) = $100")
print("=" * 60)

business_costs = {}
for name, model in {**trained_models, **tuned_models}.items():
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    cost, fn, fp = calculate_business_cost(y_test, y_pred)
    business_costs[name] = {'Total Cost': cost, 'FN': fn, 'FP': fp}
    print(f"\n📊 {name}:")
    print(f"   False Negatives: {fn:4d} × $500 = ${fn * 500:>8,}")
    print(f"   False Positives: {fp:4d} × $100 = ${fp * 100:>8,}")
    print(f"   TOTAL COST:      ${cost:>8,}")

# Find most cost-effective model
cost_df = pd.DataFrame(business_costs).T.sort_values('Total Cost')
print("\n" + "=" * 60)
print("🏆 MOST COST-EFFECTIVE MODEL:")
print(f"   {cost_df.index[0]} with total cost of ${cost_df.iloc[0]['Total Cost']:,}")
print("=" * 60)

In [ ]:
# ✅ Solution 3: Tune CatBoost with Optuna (50 trials)
print("🔧 Tuning CatBoost with Optuna (50 trials)...")

def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_state': 42,
        'verbose': False,
        'auto_class_weights': 'Balanced'
    }
    
    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, 
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study_cb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_cb.optimize(objective_catboost, n_trials=50, show_progress_bar=True)

print(f"\n✅ CatBoost Tuning Complete!")
print(f"   Best ROC-AUC (CV): {study_cb.best_value:.4f}")
print(f"   Best Parameters: {study_cb.best_params}")

# Train and evaluate tuned CatBoost
best_cb = CatBoostClassifier(**study_cb.best_params, random_state=42, verbose=False, auto_class_weights='Balanced')
best_cb.fit(X_train, y_train)
cb_pred = best_cb.predict(X_test)
cb_prob = best_cb.predict_proba(X_test)[:, 1]

cb_metrics = {
    'Accuracy': accuracy_score(y_test, cb_pred),
    'Precision': precision_score(y_test, cb_pred, zero_division=0),
    'Recall': recall_score(y_test, cb_pred, zero_division=0),
    'F1-Score': f1_score(y_test, cb_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, cb_prob)
}

print(f"\n📊 Tuned CatBoost Test Performance:")
for k, v in cb_metrics.items():
    print(f"   {k:12s}: {v:.4f}")

print(f"\n🏆 Comparison:")
print(f"   Tuned CatBoost ROC-AUC: {cb_metrics['ROC-AUC']:.4f}")
print(f"   Current Best ROC-AUC:   {all_results_df.loc[best_overall_name, 'ROC-AUC']:.4f}")

In [ ]:
# ✅ Solution 4: Voting Ensemble (Soft Voting)
from sklearn.ensemble import VotingClassifier

print("🤝 Building Voting Ensemble with Top 3 Models...")

# Select top 3 models by ROC-AUC
top_3_names = all_results_df.head(3).index.tolist()
print(f"Top 3 models for ensemble: {top_3_names}")

# Prepare estimators list
estimators = []
for name in top_3_names:
    model = tuned_models.get(name, trained_models.get(name))
    # Clean name for sklearn
    clean_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    estimators.append((clean_name, model))

# Create voting classifier (soft voting)
voting_clf = VotingClassifier(estimators=estimators, voting='soft')

# Note: VotingClassifier requires all models to use same data format
# We'll use unscaled data and retrain Logistic Regression inside the ensemble
print("\n⚠️  Note: Retraining ensemble components on consistent data format...")

# Simplified approach: use top tree-based models only for soft voting
tree_estimators = []
for name in top_3_names:
    if name != 'Logistic Regression':
        model = tuned_models.get(name, trained_models.get(name))
        clean_name = name.replace(' ', '_').replace('(', '').replace(')', '')
        tree_estimators.append((clean_name, model))

if len(tree_estimators) >= 2:
    voting_clf = VotingClassifier(estimators=tree_estimators, voting='soft')
    voting_clf.fit(X_train, y_train)
    
    ensemble_pred = voting_clf.predict(X_test)
    ensemble_prob = voting_clf.predict_proba(X_test)[:, 1]
    
    ensemble_metrics = {
        'Accuracy': accuracy_score(y_test, ensemble_pred),
        'Precision': precision_score(y_test, ensemble_pred, zero_division=0),
        'Recall': recall_score(y_test, ensemble_pred, zero_division=0),
        'F1-Score': f1_score(y_test, ensemble_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, ensemble_prob)
    }
    
    print("\n✅ Ensemble Results:")
    for k, v in ensemble_metrics.items():
        print(f"   {k:12s}: {v:.4f}")
    
    print(f"\n🏆 Ensemble vs Best Individual:")
    print(f"   Ensemble ROC-AUC: {ensemble_metrics['ROC-AUC']:.4f}")
    print(f"   Best Individual:    {all_results_df.loc[best_overall_name, 'ROC-AUC']:.4f}")
    
    if ensemble_metrics['ROC-AUC'] > all_results_df.loc[best_overall_name, 'ROC-AUC']:
        print("   🎉 ENSEMBLE WINS! Soft voting improved performance!")
    else:
        print("   💡 Individual model still outperforms. Ensembles don't always win!")
else:
    print("⚠️  Need at least 2 tree-based models for ensemble. Please check model selection.")

---

## 🏆 Summary & Key Takeaways

Congratulations! You have successfully completed **Day 81** of your 369-day Python & AI Learning Path and built a **production-ready churn prediction pipeline** from engineered features to tuned models.

### 📚 What You Accomplished Today:

✅ **Data Preparation** — Encoded categoricals, scaled numerics, and performed stratified splits  
✅ **Baseline Modeling** — Trained and compared 5 powerful algorithms (Logistic Regression, Random Forest, XGBoost, LightGBM, CatBoost)  
✅ **Comprehensive Evaluation** — Analyzed Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrices, and ROC/PR curves  
✅ **Hyperparameter Tuning** — Used Optuna's Bayesian optimization to push model performance beyond defaults  
✅ **Feature Importance** — Identified which Day 80 engineered features drive predictions most strongly  
✅ **Cross-Validation** — Validated model stability with 5-fold stratified CV  
✅ **Business Metrics** — Calculated custom cost-based evaluation for real-world decision-making  

### 🧠 Critical Insights:

🔹 **Tree-based models dominate** — XGBoost, LightGBM, and CatBoost typically outperform linear baselines on structured data  
🔹 **Hyperparameter tuning matters** — Optuna often squeezes out 2-5% additional ROC-AUC, which translates to significant business value  
🔹 **Feature engineering pays dividends** — Day 80's tenure, ratio, and text features consistently rank at the top of importance lists  
🔹 **Ensembles are powerful but not guaranteed** — Soft voting can help, but a well-tuned individual model is often sufficient  
🔹 **Business context shapes metrics** — ROC-AUC is great, but custom cost metrics align ML with real business priorities  

### 🚀 Tomorrow (Day 82):

**Tomorrow we will deploy our best model into a production-ready prediction pipeline!** We will cover:
- Saving and loading trained models with joblib/pickle
- Building a real-time prediction API with FastAPI
- Model monitoring and drift detection concepts
- Creating an interactive dashboard for business stakeholders

You now have a **tuned, validated, and interpretable churn prediction model** ready for the real world. The journey from raw data to deployed AI is nearly complete!

---

*Python & AI Learning Path | Day 81 / 369* ⭐  
*Train hard, tune smart, deploy with confidence!* 💪🚀